In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib
import warnings
warnings.filterwarnings('ignore')

In [23]:
df=pd.read_csv(r'C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\final_processed_data.csv')

In [24]:
print(df.shape)
print(df.columns)
df.head()

(33029, 30)
Index(['Unnamed: 0', 'match_obj_id', 'inningNumber', 'oversActual',
       'pitchLine', 'pitchLength', 'isFour', 'isSix', 'isWicket', 'byes',
       'legbyes', 'wides', 'noballs', 'run', 'totalRuns', 'totalWickets',
       'shotType', 'time_of_day', 'Ground Name', 'Batsman_Name',
       'Batsman_Role', 'Full Name', 'Batsman_Batting_Style',
       'Batsman_Playing_Role', 'Bowler_Name', 'Bowler_Role',
       'Full Name_bowler', 'Batting Style (s)_bowler', 'Bowler_Bowling_Style',
       'Bowler_Playing_Role'],
      dtype='object')


,Unnamed: 0,match_obj_id,inningNumber,oversActual,pitchLine,pitchLength,isFour,isSix,isWicket,byes,...,Batsman_Role,Full Name,Batsman_Batting_Style,Batsman_Playing_Role,Bowler_Name,Bowler_Role,Full Name_bowler,Batting Style (s)_bowler,Bowler_Bowling_Style,Bowler_Playing_Role
0,0,1273712,1,0.1,ON_THE_STUMPS,GOOD_LENGTH,False,False,False,0,...,P,Tony Ura,right-hand bat,opening batter,Bilal Khan,P,Bilal Khan,lhb,left-arm medium-fast,bowler
1,1,1273712,1,0.2,ON_THE_STUMPS,SHORT_OF_A_GOOD_LENGTH,False,False,False,0,...,P,Tony Ura,right-hand bat,opening batter,Bilal Khan,P,Bilal Khan,lhb,left-arm medium-fast,bowler
2,2,1273712,1,0.3,ON_THE_STUMPS,GOOD_LENGTH,False,False,False,0,...,P,Tony Ura,right-hand bat,opening batter,Bilal Khan,P,Bilal Khan,lhb,left-arm medium-fast,bowler
3,3,1273712,1,0.4,ON_THE_STUMPS,GOOD_LENGTH,False,False,False,0,...,P,Tony Ura,right-hand bat,opening batter,Bilal Khan,P,Bilal Khan,lhb,left-arm medium-fast,bowler
4,4,1273712,1,0.5,ON_THE_STUMPS,GOOD_LENGTH,False,False,True,0,...,P,Tony Ura,right-hand bat,opening batter,Bilal Khan,P,Bilal Khan,lhb,left-arm medium-fast,bowler


In [25]:
df.columns

Index(['Unnamed: 0', 'match_obj_id', 'inningNumber', 'oversActual',
       'pitchLine', 'pitchLength', 'isFour', 'isSix', 'isWicket', 'byes',
       'legbyes', 'wides', 'noballs', 'run', 'totalRuns', 'totalWickets',
       'shotType', 'time_of_day', 'Ground Name', 'Batsman_Name',
       'Batsman_Role', 'Full Name', 'Batsman_Batting_Style',
       'Batsman_Playing_Role', 'Bowler_Name', 'Bowler_Role',
       'Full Name_bowler', 'Batting Style (s)_bowler', 'Bowler_Bowling_Style',
       'Bowler_Playing_Role'],
      dtype='object')

In [26]:

df['isWicket'] = df['isWicket'].astype(int)
df['isBoundary'] = df['isFour'] | df['isSix']
df['isBoundary'] = df['isBoundary'].astype(int)
# Calculate fundamental aggregates for each bowler
bowler_stats = df.groupby(['Bowler_Name', 'Bowler_Bowling_Style']).agg(
    balls_bowled=('Bowler_Name', 'count'),
    total_runs=('run', 'sum'),
    total_boundaries=('isBoundary', 'sum'),
    total_wickets=('isWicket', 'sum'),
).reset_index()

# Derived metrics
bowler_stats['dot_balls'] = df[df['run'] == 0].groupby('Bowler_Name')['run'].count().reindex(bowler_stats['Bowler_Name']).fillna(0).astype(int).values
bowler_stats['economy_rate'] = bowler_stats['total_runs'] / (bowler_stats['balls_bowled'] / 6)
bowler_stats['strike_rate'] = np.where(bowler_stats['total_wickets'] > 0, bowler_stats['balls_bowled'] / bowler_stats['total_wickets'], np.nan)
bowler_stats['boundary_percentage'] = (bowler_stats['total_boundaries'] / bowler_stats['balls_bowled']) * 100
bowler_stats['dot_ball_percentage'] = (bowler_stats['dot_balls'] / bowler_stats['balls_bowled']) * 100
bowler_stats['wicket_percentage'] = (bowler_stats['total_wickets'] / bowler_stats['balls_bowled']) * 100

# Clean up possible infinities / NaN
bowler_stats.replace([np.inf, -np.inf], np.nan, inplace=True)
bowler_stats.fillna(0, inplace=True)

# Success Index (weighted composite metric)
bowler_stats['success_index'] = (
    (100 - bowler_stats['economy_rate']) * 0.3 +
    bowler_stats['dot_ball_percentage'] * 0.3 +
    bowler_stats['wicket_percentage'] * 0.4
)

# Sort by success index descending
bowler_stats.sort_values(by='success_index', ascending=False, inplace=True)

print(" Bowler stats computed successfully.")
bowler_stats.head(10)


 Bowler stats computed successfully.


,Bowler_Name,Bowler_Bowling_Style,balls_bowled,total_runs,total_boundaries,total_wickets,dot_balls,economy_rate,strike_rate,boundary_percentage,dot_ball_percentage,wicket_percentage,success_index
231,OEG Baartman,right-arm medium-fast,74,48,5,5,47,3.891892,14.800000,6.756757,63.513514,6.756757,50.589189
88,DS Airee,"right-arm medium,right-arm offbreak",12,6,0,1,7,3.000000,12.000000,0.000000,58.333333,8.333333,49.933333
98,Fazalhaq Farooqi,left-arm fast-medium,132,122,14,11,73,5.545455,12.000000,10.606061,55.303030,8.333333,48.260606
4,A Nortje,right-arm fast,300,258,23,33,154,5.160000,9.090909,7.666667,51.333333,11.000000,48.252000
219,NL McCullum,right-arm offbreak,31,21,1,3,16,4.064516,10.333333,3.225806,51.612903,9.677419,48.135484
303,Tanzim Hasan Sakib,right-arm medium,68,67,9,5,38,5.911765,13.600000,13.235294,55.882353,7.352941,47.932353
173,M Jansen,left-arm medium-fast,72,53,5,3,42,4.416667,24.000000,6.944444,58.333333,4.166667,47.841667
306,VJ Kingma,right-arm medium-fast,64,57,7,3,37,5.343750,21.333333,10.937500,57.812500,4.687500,47.615625
125,JC Archer,right-arm fast,57,52,6,4,31,5.473684,14.250000,10.526316,54.385965,7.017544,47.480702
48,BD Glover,right-arm fast,94,94,11,9,48,6.000000,10.444444,11.702128,51.063830,9.574468,47.348936


In [31]:
# --- Cell 3: Normalize, Enhance & Export Bowler Stats ---

from sklearn.preprocessing import MinMaxScaler

# Select the metrics we want to normalize
metrics = ['economy_rate', 'strike_rate', 'boundary_percentage',
           'dot_ball_percentage', 'wicket_percentage', 'success_index']

scaler = MinMaxScaler()
bowler_stats_scaled = bowler_stats.copy()
bowler_stats_scaled[metrics] = scaler.fit_transform(bowler_stats_scaled[metrics])

# Add bowler strength zone for quick interpretation
def categorize_strength(row):
    if row['success_index'] >= 0.75:
        return 'Elite'
    elif row['success_index'] >= 0.5:
        return 'Strong'
    elif row['success_index'] >= 0.25:
        return 'Average'
    else:
        return 'Weak'

bowler_stats_scaled['bowler_strength_zone'] = bowler_stats_scaled.apply(categorize_strength, axis=1)

# Reorder columns for clarity
cols_order = ['Bowler_Name', 'Bowler_Bowling_Style', 'balls_bowled', 'total_runs', 'total_boundaries', 
              'total_wickets', 'dot_balls', 'economy_rate', 'strike_rate', 
              'boundary_percentage', 'dot_ball_percentage', 'wicket_percentage', 
              'success_index', 'bowler_strength_zone']
bowler_stats_scaled = bowler_stats_scaled[cols_order]

# Save enhanced stats
bowler_stats_scaled.to_csv('bowler_stats.csv', index=False)

print(" Bowler stats normalized and saved successfully.")
print("File: bowler_stats.csv")
bowler_stats_scaled.head(10)


 Bowler stats normalized and saved successfully.
File: bowler_stats.csv


,Bowler_Name,Bowler_Bowling_Style,balls_bowled,total_runs,total_boundaries,total_wickets,dot_balls,economy_rate,strike_rate,boundary_percentage,dot_ball_percentage,wicket_percentage,success_index,bowler_strength_zone
231,OEG Baartman,right-arm medium-fast,74,48,5,5,47,0.046942,0.202740,0.090090,1.000000,0.270270,1.000000,Elite
88,DS Airee,"right-arm medium,right-arm offbreak",12,6,0,1,7,0.000000,0.164384,0.000000,0.918440,0.333333,0.973859,Elite
98,Fazalhaq Farooqi,left-arm fast-medium,132,122,14,11,73,0.133971,0.164384,0.141414,0.870729,0.333333,0.907188,Elite
4,A Nortje,right-arm fast,300,258,23,33,154,0.113684,0.124533,0.102222,0.808227,0.440000,0.906845,Elite
219,NL McCullum,right-arm offbreak,31,21,1,3,16,0.056027,0.141553,0.043011,0.812629,0.387097,0.902201,Elite
303,Tanzim Hasan Sakib,right-arm medium,68,67,9,5,38,0.153251,0.186301,0.176471,0.879850,0.294118,0.894104,Elite
173,M Jansen,left-arm medium-fast,72,53,5,3,42,0.074561,0.328767,0.092593,0.918440,0.166667,0.890490,Elite
306,VJ Kingma,right-arm medium-fast,64,57,7,3,37,0.123355,0.292237,0.145833,0.910239,0.187500,0.881480,Elite
125,JC Archer,right-arm fast,57,52,6,4,31,0.130194,0.195205,0.140351,0.856290,0.280702,0.876103,Elite
48,BD Glover,right-arm fast,94,94,11,9,48,0.157895,0.143075,0.156028,0.803984,0.382979,0.870851,Elite


In [32]:


# Ensure proper types
df['isWicket'] = df['isWicket'].astype(int)
df['isBoundary'] = df['isBoundary'].astype(int)
df['run'] = df['run'].fillna(0).astype(int)

# Aggregate per (Bowler, Batsman) pair
bowling_success = df.groupby(['Bowler_Name', 'Bowler_Bowling_Style', 'Batsman_Name']).agg(
    balls_bowled=('Bowler_Name', 'count'),
    total_runs=('run', 'sum'),
    total_boundaries=('isBoundary', 'sum'),
    total_wickets=('isWicket', 'sum'),
    dot_balls=('run', lambda x: (x == 0).sum())
).reset_index()

# Derived metrics per batsman-bowler pair
bowling_success['economy_rate'] = bowling_success['total_runs'] / (bowling_success['balls_bowled'] / 6)
bowling_success['strike_rate'] = np.where(bowling_success['total_wickets'] > 0,
                                          bowling_success['balls_bowled'] / bowling_success['total_wickets'], np.nan)
bowling_success['boundary_percentage'] = (bowling_success['total_boundaries'] / bowling_success['balls_bowled']) * 100
bowling_success['dot_ball_percentage'] = (bowling_success['dot_balls'] / bowling_success['balls_bowled']) * 100
bowling_success['wicket_percentage'] = (bowling_success['total_wickets'] / bowling_success['balls_bowled']) * 100

# Clean up
bowling_success.replace([np.inf, -np.inf], np.nan, inplace=True)
bowling_success.fillna(0, inplace=True)

# Calculate pairwise success index
bowling_success['success_index'] = (
    (100 - bowling_success['economy_rate']) * 0.3 +
    bowling_success['dot_ball_percentage'] * 0.3 +
    bowling_success['wicket_percentage'] * 0.4
)

# Normalize success_index for comparability
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
bowling_success['success_index_scaled'] = scaler.fit_transform(bowling_success[['success_index']])

# Add quick performance tag
def performance_label(x):
    if x >= 0.75:
        return 'High Success'
    elif x >= 0.5:
        return 'Moderate Success'
    else:
        return 'Low Success'

bowling_success['performance_label'] = bowling_success['success_index_scaled'].apply(performance_label)

# Sort by success index
bowling_success.sort_values(by='success_index_scaled', ascending=False, inplace=True)

# Save output
bowling_success.to_csv('bowling_success_model.csv', index=False)

print(" Bowling Success Model built successfully.")
print("Shape:", bowling_success.shape)
bowling_success.tail(10)


 Bowling Success Model built successfully.
Shape: (6971, 16)


,Bowler_Name,Bowler_Bowling_Style,Batsman_Name,balls_bowled,total_runs,total_boundaries,total_wickets,dot_balls,economy_rate,strike_rate,boundary_percentage,dot_ball_percentage,wicket_percentage,success_index,success_index_scaled,performance_label
6734,TS Mills,left-arm fast,PJ Cummins,2,12,2,0,0,36.0,0.0,100.0,0.0,0.0,19.2,0.0,Low Success
2297,HH Pandya,right-arm medium-fast,PM Nevill,1,6,1,0,0,36.0,0.0,100.0,0.0,0.0,19.2,0.0,Low Success
4576,Mohammad Wasim,right-arm fast-medium,Soumya Sarkar,1,6,1,0,0,36.0,0.0,100.0,0.0,0.0,19.2,0.0,Low Success
1126,BTJ Wheal,right-arm fast-medium,KV Morea,1,6,1,0,0,36.0,0.0,100.0,0.0,0.0,19.2,0.0,Low Success
656,Amir Hamza,slow left-arm orthodox,DJ Willey,2,12,2,0,0,36.0,0.0,100.0,0.0,0.0,19.2,0.0,Low Success
4750,Mustafizur Rahman,left-arm fast-medium,MJ McClenaghan,1,6,1,0,0,36.0,0.0,100.0,0.0,0.0,19.2,0.0,Low Success
4830,NLTC Perera,right-arm medium-fast,BA Stokes,1,6,1,0,0,36.0,0.0,100.0,0.0,0.0,19.2,0.0,Low Success
3559,LH Ferguson,right-arm fast,SM Curran,1,6,1,0,0,36.0,0.0,100.0,0.0,0.0,19.2,0.0,Low Success
900,BA Stokes,right-arm fast-medium,CR Brathwaite,4,24,4,0,0,36.0,0.0,100.0,0.0,0.0,19.2,0.0,Low Success
4887,Nadeem Ahmed,slow left-arm orthodox,MW Machan,1,6,1,0,0,36.0,0.0,100.0,0.0,0.0,19.2,0.0,Low Success


In [ ]:

import plotly.express as px
import plotly.graph_objects as go

# Load the generated success model file (if not already in memory)
bowling_success = pd.read_csv('bowling_success_model.csv')

print(" Bowling success data loaded:", bowling_success.shape)

# Ensure proper sorting for display
bowling_success.sort_values(by='success_index_scaled', ascending=False, inplace=True)

#  Function: Top Bowlers vs a Specific Batsman ---
def show_top_bowlers_vs_batsman(batsman_query, top_n=10):
    # Case-insensitive partial matching
    mask = bowling_success['Batsman_Name'].str.lower().str.contains(batsman_query.lower(), na=False)
    subset = bowling_success[mask]
    
    if subset.empty:
        # Suggest similar names if not found
        possible_matches = [name for name in bowling_success['Batsman_Name'].unique() if batsman_query.lower() in name.lower()]
        if possible_matches:
            print(f"No exact match, but similar names found: {possible_matches[:5]}")
        else:
            print(f"No data found for '{batsman_query}'. Try a partial name (e.g., 'kohli' instead of 'Virat Kohli').")
        return
    
    fig = px.bar(
        subset.head(top_n),
        x='Bowler_Name',
        y='success_index_scaled',
        color='Bowler_Bowling_Style',
        hover_data=['economy_rate', 'wicket_percentage', 'dot_ball_percentage'],
        title=f"Top {top_n} Bowlers vs {subset['Batsman_Name'].iloc[0]} (by Success Index)",
        color_discrete_sequence=px.colors.qualitative.Vivid
    )
    fig.update_layout(xaxis_title='Bowler', yaxis_title='Scaled Success Index (0–1)')
    fig.show()

# Try again with partial matching
show_top_bowlers_vs_batsman("kohli")
# Bowling Style Performance Comparison 
def show_bowling_style_comparison():
    fig = px.box(
        bowling_success,
        x='Bowler_Bowling_Style',
        y='success_index_scaled',
        color='Bowler_Bowling_Style',
        title='Bowling Style Effectiveness (Success Index Distribution)',
        points='all'
    )
    fig.update_layout(xaxis_title='Bowling Style', yaxis_title='Scaled Success Index (0–1)', showlegend=False)
    fig.show()

#  Economy vs Wicket% Scatter ---
def show_economy_vs_wickets():
    fig = px.scatter(
        bowling_success,
        x='economy_rate',
        y='wicket_percentage',
        size='dot_ball_percentage',
        color='success_index_scaled',
        hover_name='Bowler_Name',
        title='Economy Rate vs Wicket% (Bubble = Dot Ball%)',
        color_continuous_scale='RdYlGn'
    )
    fig.update_layout(xaxis_title='Economy Rate', yaxis_title='Wicket Percentage')
    fig.show()


show_bowling_style_comparison()
show_economy_vs_wickets()


 Bowling success data loaded: (6971, 16)


In [30]:
show_top_bowlers_vs_batsman("babar")
